In [1]:
# Global imports
import biosteam as bst, thermosteam as tmo, biorefineries as bf, numpy as np, pandas as pd
from biorefineries import cellulosic
from biosteam import main_flowsheet as F, units

# Local imports
from atj_saf.atj_bst.etj_chemicals import create_chemicals
from atj_saf.atj_bst.etj_settings import feed_parameters, dehyd_data, olig_data, prod_selectivity, hydgn_data, price_data, h2_recovery
from atj_saf.atj_bst.etj_utils import calculate_ethanol_flow
from atj_saf.atj_bst.atj_bst_units import AdiabaticReactor, IsothermalReactor, EthanolStorageTank, HydrocarbonProductTank, HydrogenStorageTank, CatalystMixer
bst.F.set_flowsheet('etj') # F is the main flowsheet
etj_chems = create_chemicals()
bst.settings.set_thermo(etj_chems)
bst.settings.CEPCI = 800.8 # For the year 2023 from https://personalpages.manchester.ac.uk/staff/tom.rodgers/Interactive_graphs/CEPCI.html?reactors/CEPCI/index.html

def create_etj_system_no_facilities(ins=None):


    etoh_flow = calculate_ethanol_flow(9)

    # Bioethanol feed — use caller-supplied stream or create from settings
    if ins is None:
        etoh_in = bst.Stream(
            'Ethanol_In',
            Ethanol = etoh_flow,
            Water =  etoh_flow*((1-feed_parameters['purity'])/(feed_parameters['purity'])),
            units = 'kg/hr',
            T = feed_parameters['temperature'],
            P = feed_parameters['pressure'],
            phase = feed_parameters['phase'])
    else:
        etoh_in = ins


    # Reactions

    #1) Gas phase dehydration of ethanol to ethylene
    dehydration_rxn = bst.Reaction('Ethanol,g -> Water,g + Ethylene,g', reactant = 'Ethanol',
                                X = dehyd_data['conv'], phases = 'lg',  basis = 'mol')


    #2) Ethylene oligomerization to olefins in gas and liquid phase
    oligomerization_rxn = bst.ParallelReaction([
    bst.Reaction('2Ethylene,g -> Butene,g',            reactant = 'Ethylene',     X = olig_data['conv']*prod_selectivity['C4H8'],    basis = 'wt',  phases = 'lg',  correct_atomic_balance = True),
    bst.Reaction('3Ethylene,g -> Hex-1-ene,g',       reactant = 'Ethylene',     X = olig_data['conv']*prod_selectivity['C6H12'],   basis = 'wt',  phases = 'lg',  correct_atomic_balance = True),
    bst.Reaction('5Ethylene,g -> Dec-1-ene,l',         reactant = 'Ethylene',     X = olig_data['conv']*prod_selectivity['C10H20'],  basis = 'wt',  phases = 'lg',  correct_atomic_balance = True),
    bst.Reaction('9Ethylene,g -> Octadec-1-ene,l',     reactant = 'Ethylene',     X = olig_data['conv']*prod_selectivity['C18H36'],  basis = 'wt',  phases = 'lg',  correct_atomic_balance = True)])


    hydrogenation_rxn = bst.ParallelReaction([
    bst.Reaction('Butene,g + Hydrogen,g -> Butane,g',               reactant = 'Butene',          X = hydgn_data['conv'],  basis = 'mol',  phases = 'lg',  correct_atomic_balance = True),
    bst.Reaction('Butene,l + Hydrogen,g -> Butane,l',               reactant = 'Butene',          X = hydgn_data['conv'],  basis = 'mol',  phases = 'lg',  correct_atomic_balance = True),
    bst.Reaction('Hex-1-ene,g + Hydrogen,g -> Hexane,g',            reactant = 'Hex-1-ene',       X = hydgn_data['conv'],  basis = 'mol',  phases = 'lg',  correct_atomic_balance = True),
    bst.Reaction('Hex-1-ene,l + Hydrogen,g -> Hexane,l',            reactant = 'Hex-1-ene',       X = hydgn_data['conv'],  basis = 'mol',  phases = 'lg',  correct_atomic_balance = True),
    bst.Reaction('Dec-1-ene,l + Hydrogen,g -> Decane,l',            reactant = 'Dec-1-ene',       X = hydgn_data['conv'],  basis = 'mol',  phases = 'lg',  correct_atomic_balance = True),
    bst.Reaction('Dec-1-ene,g + Hydrogen,g -> Decane,g',            reactant = 'Dec-1-ene',       X = hydgn_data['conv'],  basis = 'mol',  phases = 'lg',  correct_atomic_balance = True),
    bst.Reaction('Octadec-1-ene,l + Hydrogen,g -> Octadecane,l',    reactant = 'Octadec-1-ene',   X = hydgn_data['conv'],  basis = 'mol',  phases = 'lg',  correct_atomic_balance = True),
    bst.Reaction('Octadec-1-ene,g + Hydrogen,g -> Octadecane,g',    reactant = 'Octadec-1-ene',   X = hydgn_data['conv'],  basis = 'mol',  phases = 'lg',  correct_atomic_balance = True)])


    # Recycle streams
    dehyd_recycle = bst.MultiStream('dehyd_recycle', phases = ('g','l'))         # Unreacted ethanol
    ethylene_recycle = bst.MultiStream('ethylene_recycle', phases = ('g','l'))   # Unreacted ethylene
    h2_recycle= bst.Stream(ID = 'h2_recycle', P = 3e6, phase = 'g')              # Excess hydrogen

    # Catalyst replacement streams — flows updated each iteration via add_specification on each reactor
    syndol_replacement    = bst.Stream('Dehyd_cat_replacement', phase = 's')
    ni_si_al_replacement  = bst.Stream('Olig_cat_replacement',  phase = 's')
    co_mo_replacement     = bst.Stream('Hydgn_cat_replacement', phase = 's')


    # Area 100: Feed Storage
    etoh_storage = EthanolStorageTank('T101', ins = etoh_in)

    h2_in = bst.Stream(ID = 'Hydrogen_In', P = 3e6, phase = 'g')
    h2_storage = HydrogenStorageTank('T102', ins = h2_in)


    # Area 200: Catalytic Upgrading
    pump_1 = bst.Pump('P201', ins = etoh_storage.outs[0], P = 1373000)

    furnace_1 = bst.HXutility('H201', ins = pump_1.outs[0], T = 500, rigorous = True)

    mixer_1 = bst.Mixer('M201', ins = (furnace_1.outs[0], dehyd_recycle), rigorous = True)

    furnace_2 = bst.HXutility('H202', ins = mixer_1.outs[0], T = 481 + 273.15, rigorous = True)

    dehyd_1 = AdiabaticReactor('R201', ins = furnace_2.outs[0],
                            conversion = dehyd_data['conv'],
                            temperature = dehyd_data['temp'],
                            pressure = dehyd_data['pressure'],
                            WHSV = dehyd_data['whsv'],
                            vessel_type = 'Vertical',
                            vessel_material = 'Stainless steel 316',
                            catalyst_price=price_data['dehydration_catalyst'],
                            catalyst_lifetime = dehyd_data['catalyst_lifetime'],
                            reaction = dehydration_rxn)

    @dehyd_1.add_specification(run = True)
    def update_syndol_flow():
        # Catalyst weight = feed_flow / WHSV — identical to AdiabaticReactor._design() formula
        cat_wt = dehyd_1.ins[0].F_mass / dehyd_data['whsv']
        syndol_replacement.imass['Syndol'] = cat_wt / dehyd_data['catalyst_lifetime'] / 8760  # kg/hr


    splitter_1 = bst.Splitter('S201', ins = dehyd_1.outs[0], outs = ('flash_in', dehyd_recycle), split = 0.3)

    flash_1 = bst.Flash('T201', ins = splitter_1.outs[0], outs = ('ETHYLENE_WATER', 'WW_1'), T= 420,  P = 1.063e6)

    comp_1 = bst.IsentropicCompressor('K201', ins = flash_1.outs[0], P = 2e6, vle = True, eta = 0.72, driver_efficiency = 1)

    distillation_1 = bst.BinaryDistillation('D201', ins = comp_1.outs[0],
                                                outs = ('ethylene_water', 'WW'),
                                    LHK = ('Ethylene', 'Water'),
                                    P = 2e+06,
                                    y_top = 0.999, x_bot = 0.001, k = 2,
                                    is_divided = False)

    comp_2 = bst.IsentropicCompressor('K202', ins = distillation_1.outs[0], P = olig_data['pressure'], vle = True, eta = 0.72, driver_efficiency = 1)

    distillation_2 = bst.BinaryDistillation('D202', ins = comp_2.outs[0],
                                    LHK = ('Ethylene', 'Ethanol'),
                                    P = 3.5e+06,
                                    y_top = 0.9999, x_bot = 0.0001, k = 2,
                                    is_divided = False)

    cooler_3 = bst.HXutility('H203', ins = distillation_2.outs[0], T = 393.15, rigorous = True)

    mixer_2 = bst.Mixer(ID = 'M202', ins = (cooler_3.outs[0],ethylene_recycle), rigorous = True)

    olig_1 = IsothermalReactor('R202', ins = mixer_2.outs[0],
                                conversion = olig_data['conv'],
                                temperature = olig_data['temp'],
                                pressure = olig_data['pressure'],
                                WHSV = olig_data['whsv'],
                                catalyst_price = price_data['oligomerization_catalyst'],
                            reaction = oligomerization_rxn)

    @olig_1.add_specification(run = True)
    def update_ni_si_al_flow():
        cat_wt = olig_1.ins[0].F_mass / olig_data['whsv']
        ni_si_al_replacement.imass['Nickel_SiAl'] = cat_wt / olig_data['catalyst_lifetime'] / 8760  # kg/hr


    splitter_2 = bst.Splitter('S202', ins = olig_1.outs[0], outs = (ethylene_recycle,'oligs'),  split = {'Ethylene':1.0})

    # 3:1 excess hydrogen to oligomers molar ratio, with 100% molar conversion 2x moles oligomer H2 is left,
    # and 85 mol% is recovered, so fresh H2 must cover reacted H2 (1x moles oligomer) and PSA losses (0.15 * 2x moles oligomer)
    @h2_storage.add_specification(run = True)
    def h2_flow():
        total_h2_req = 3 * (olig_1.outs[0].imol['Butene'] + olig_1.outs[0].imol['Hex-1-ene']
                            + olig_1.outs[0].imol['Dec-1-ene'] + olig_1.outs[0].imol['Octadec-1-ene'])
        h2_storage.ins[0].imol['Hydrogen'] = total_h2_req - h2_recycle.imol['Hydrogen']


    mixer_4 = bst.Mixer('M203', ins = (h2_storage.outs[0], splitter_2.outs[1], h2_recycle), rigorous = True)

    furnace_3 = bst.HXutility('H204', mixer_4.outs[0], T = 350 +273.15, rigorous = True)

    hydgn_1 = AdiabaticReactor('R203', ins = furnace_3.outs[0],
                            conversion = hydgn_data['conv'],
                            temperature = hydgn_data['temp'],
                            pressure = hydgn_data['pressure'],
                            WHSV = hydgn_data['whsv'],
                            catalyst_price = price_data['hydrogenation_catalyst'],
                            reaction = hydrogenation_rxn)

    @hydgn_1.add_specification(run = True)
    def update_co_mo_flow():
        cat_wt = hydgn_1.ins[0].F_mass / hydgn_data['whsv']
        co_mo_replacement.imass['CobaltMolybdenum'] = cat_wt / hydgn_data['catalyst_lifetime'] / 8760  # kg/hr


    cooler_5 = bst.HXutility('H205', ins = hydgn_1.outs[0], T = 250, rigorous = True)

    flash_2 = bst.Flash('T202', ins = cooler_5-0, T = 250, P = 5e5)

    psa_splitter = bst.Splitter('S203', ins = flash_2-0, outs = (h2_recycle,'BT_feed'),  split = {'Hydrogen':h2_recovery})


    # Area 300: Product Fractionation
    distillation_3 = bst.BinaryDistillation('D301', ins = flash_2.outs[1],
                                    outs = ('distillate', 'bottoms'),
                                    LHK = ('Hexane', 'Decane'),
                                    y_top = 0.99, x_bot = 0.01, k = 2,
                                    is_divided = True)

    distillation_4 = bst.BinaryDistillation('D302', ins = distillation_3.outs[1],
                                    outs = ('distillate_1', 'bottoms_1'),
                                    LHK = ('Decane', 'Octadecane'),
                                    y_top = 0.99, x_bot = 0.01, k = 2,
                                    is_divided = True)

    cooler_6 = bst.HXutility('H301', ins = distillation_3.outs[0], V = 0, rigorous = True)

    cooler_7 = bst.HXutility('H302', ins = distillation_4.outs[0], T = 15+273.15, rigorous = True)

    # rigorous=True VLE can produce a trace gas fraction at 15°C; override to liquid after each run
    # so HydrocarbonProductTank always receives a single-phase liquid stream
    @cooler_7.add_specification(run = False)
    def simulate_cooler_7():
        cooler_7._run()
        cooler_7._design()
        cooler_7._cost()
        cooler_7.outs[0].phase = 'l'

    cooler_8 = bst.HXutility('H303', ins = distillation_4.outs[1], T = 15+273.15, rigorous = True)

    @cooler_8.add_specification(run = False)
    def simulate_cooler_8():
        cooler_8._run()
        cooler_8._design()
        cooler_8._cost()
        cooler_8.outs[0].phase = 'l'


    # Area 500: Product Storage
    rn_storage  = HydrocarbonProductTank('T501', ins = cooler_6.outs[0], outs = 'RN')
    saf_storage = HydrocarbonProductTank('T502', ins = cooler_7.outs[0], outs = 'SAF')
    rd_storage  = HydrocarbonProductTank('T503', ins = cooler_8.outs[0], outs = 'RD')


    # Area 600: Wastewater collection (no WWT facility — routed to central utilities in combined system)
    WW_mixer = bst.Mixer('M601', ins = (flash_1-1, distillation_1-1, distillation_2-1), rigorous = True)
    WW_cooler = bst.HXutility('H602', ins = WW_mixer.outs[0], V = 0, rigorous = True)

    catalyst_replacement_unit = CatalystMixer(ins = (syndol_replacement, ni_si_al_replacement, co_mo_replacement))


    etj_sys = bst.System('atj_sys', path = (etoh_storage, pump_1, furnace_1, mixer_1, furnace_2, dehyd_1, splitter_1, flash_1, comp_1,
                                            distillation_1, comp_2, distillation_2, cooler_3, mixer_2,
                                            olig_1, splitter_2, h2_storage, mixer_4, furnace_3, hydgn_1, cooler_5,
                                            flash_2, psa_splitter, distillation_3, distillation_4, cooler_6, cooler_7, cooler_8,
                                            rn_storage, saf_storage, rd_storage, WW_mixer, WW_cooler, catalyst_replacement_unit),
                                            recycle = (dehyd_recycle, ethylene_recycle, h2_recycle))

    return etj_sys


In [2]:
etj_system = create_etj_system_no_facilities()

In [ ]:
etj_system.simulate()

c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\bubble_point.py:128: RuntimeWarning: Ethylene has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\dew_point.py:129: RuntimeWarning: Ethylene has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\bubble_point.py:128: RuntimeWarning: Hydrogen has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\dew_point.py:129: RuntimeWarning: Hydrogen has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages

In [4]:
etj_system.show()

System: atj_sys
Highest convergence error among components in recycle
streams {S201-1, S202-0, S203-0} after 4 loops:
- flow rate   6.57e-01 kmol/hr (0.18%)
- temperature 2.13e-01 K (0.033%)
ins...
[0] Dehyd_cat_replacement  
    phase: 's', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Syndol  4.79
[1] Olig_cat_replacement  
    phase: 's', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Nickel_SiAl  0.35
[2] Hydgn_cat_replacement  
    phase: 's', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): CobaltMolybdenum  0.0613
[3] Ethanol_In  
    phase: 'l', T: 293.15 K, P: 101325 Pa
    flow (kmol/hr): Water    2.09
                    Ethanol  162
[4] Hydrogen_In  
    phase: 'g', T: 298.15 K, P: 3e+06 Pa
    flow (kmol/hr): Hydrogen  58.5
outs...
[0] s26  
    phase: 's', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Syndol            4.79
                    Nickel_SiAl       0.35
                    CobaltMolybdenum  0.0613
[1] BT_feed  
    phase: 'g', T: 250 K, P: 500000 Pa
    flow (kmo

In [1]:
from biosteam import main_flowsheet as F
from atj_saf.atj_bst.etj_system import create_etj_system   
from etj_settings import price_data   
from cellulosic_tea_etj import create_cellulosic_ethanol_tea

                                                                                                                                                                                                     
etj_sys =  create_etj_system(ins=None, req_saf=9)
etj_sys.simulate()
etj_sys.show()

c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\process_tools\system_factory.py:279: RuntimeWarning: <Mixer: M601> has been replaced in registry
  self.f(ins, outs, **kwargs)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\bubble_point.py:128: RuntimeWarning: Ethylene has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\dew_point.py:129: RuntimeWarning: Ethylene has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\bubble_point.py:128: RuntimeWarning: Hydrogen has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\dew_point.py:129: R

System: atj_sys
Highest convergence error among components in recycle
streams {S201-1, S202-0, S203-0} after 4 loops:
- flow rate   6.57e-01 kmol/hr (0.18%)
- temperature 2.13e-01 K (0.033%)
ins...
[0] Dehyd_cat_replacement  
    phase: 's', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Syndol  4.79
[1] Olig_cat_replacement  
    phase: 's', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Nickel_SiAl  0.35
[2] Hydgn_cat_replacement  
    phase: 's', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): CobaltMolybdenum  0.0613
[3] Ethanol_In  
    phase: 'l', T: 293.15 K, P: 101325 Pa
    flow (kmol/hr): Water    2.09
                    Ethanol  162
[4] Hydrogen_In  
    phase: 'g', T: 298.15 K, P: 3e+06 Pa
    flow (kmol/hr): Hydrogen  58.5
[5] air_lagoon  
    phase: 'g', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): N2  0.44
                    O2  0.109
[6] caustic  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water  0.948
                    NaOH   0.427
[7] -  
    phase

c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\units\design_tools\pressure_vessel.py:104: CostWarning: <RefluxDrum: reflux_drum> Vertical vessel weight (548.4 lb) is out of bounds (4200 to 1e+06 lb) for cost correlation
  return method(pressure, diameter, length)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\_unit.py:1069: RuntimeWarning: `HXutility._run` method added unit results (e.g., purchase costs, heat and power utilities); unit results should only be added in `_design` or `_cost` methods
  warn(
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\_unit.py:986: RuntimeWarning: the purchase cost item, 'Total Cost', has no defined bare-module factor in the 'HydrocarbonProductTank.F_BM' dictionary; bare-module factor now has a default value of 1
  warn(f"the purchase cost item, '{name}', has "


In [2]:

operators_per_section = 1  # operators per section from Seider recommendation
num_process_sections = 5  # number of proces sections from Seider recommendation [5 areas - storage, upgrading, separation, wwt, bt)
num_operators_per_shift = operators_per_section * num_process_sections * 1  # multiplied by 2 for large continuous flow process (e.g., 1000 ton/day product). from Seider pg 505
num_shifts = 5  # number of shifts
pay_rate = 40  # $/hr
DWandB = num_operators_per_shift * num_shifts * 2080 * pay_rate  # direct wages and benefits. DWandB [$/year] = (operators/shift)*(5 shifts)*(40 hr/week)*(operating days/year-operator)*($/hr)
Dsalaries_benefits = 0.15 * DWandB  # direct salaries and benefits from Seider
O_supplies = 0.06 * DWandB  # Operating supplies and services from Seider
technical_assistance = 5 * 75000  # $/year. Technical assistance to manufacturing. assume 5 workers at $75000/year
control_lab = 5 * 80000  # $/year. Control laboratory. assume 5 workers at $80000/year
labor = DWandB + Dsalaries_benefits + O_supplies + technical_assistance + control_lab 

    

In [3]:
F.Ethanol_In.price = price_data['ethanol']            
F.Hydrogen_In.price = price_data['hydrogen']
F.RN.price = price_data['renewable_naphtha']
saf_stream = F.SAF
F.RD.price = price_data['renewable_diesel']
F.Dehyd_cat_replacement.price = price_data['dehydration_catalyst']
F.Olig_cat_replacement.price = price_data['oligomerization_catalyst']
F.Hydgn_cat_replacement.price = price_data['hydrogenation_catalyst']


In [4]:
final_tea = create_cellulosic_ethanol_tea(etj_sys)


In [5]:
final_tea.labor_cost = labor

In [6]:
mjsp = round(((final_tea.solve_price(F.SAF)*F.SAF.rho)/264.172),2)

In [12]:
print(f'The MSP for ETJ-derived SAF is  {mjsp} USD/gal')

The MSP for ETJ-derived SAF is  9.06 USD/gal


In [8]:
print('ETJ system installed equipment cost is', round(etj_sys.installed_equipment_cost / 1e6, 2), 'MM USD')
print('ETJ system purchase equipment cost is', round(etj_sys.purchase_cost / 1e6, 2), 'MM USD')


ETJ system installed equipment cost is 58.73 MM USD
ETJ system purchase equipment cost is 44.83 MM USD


In [9]:
opex = etj_sys.utility_cost + etj_sys.material_cost
round(opex/1e6,3)

65.492

In [10]:
etj_sys.utility_cost/1e6

2.47485954014329

In [11]:
etj_sys.material_cost/1e6

63.017364552049585